# Batch-effect reverse-noise bifurcation -- seeds 10-19

The mirror image of the bifurcation benchmark: a LARGE affine batch effect for t < 4 tapering to a
small one after, so the observed snapshots start far from the truth and settle onto it -- the
estimator has to pull the path in rather than rein it out. The trunk margin is widened
(`delta_pre` 0.25 -> 0.6) so the branch order broadly survives the bigger early noise. The
trajectory grid is t = 0..8; **t = 0 is taken observed** and never estimated, so the fitted series
is `[observed t0] + est(t=1..8)`.

Data: `reverse_sim.make_reverse_noise(seed, ...)`, deterministic in `seed`, so every method sees
identical data. Same three stages as the bifurcation twin: estimate per seed (saving clouds +
generators + the shared baseline npz), fit our three trajectory methods, aggregate the metrics.
`SMOKE = 1` runs a single seed end to end.

In [ ]:
import os, sys, time, json
import numpy as np
import matplotlib.pyplot as plt


def _bootstrap():
    here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
    # this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
    for d in (here, os.path.abspath(os.path.join(here, os.pardir, os.pardir, "tools"))):
        if d not in sys.path:
            sys.path.insert(0, d)
    import _repo
    _repo.add_paths()
    return _repo


P = _bootstrap()
REPO = P.REPO
import uotreg as U
from uotreg import trajectory_metrics as TM
from uotreg.metrics import w2
from uotreg.plotting import show_noisy_data, show_estimates, plot_trajectory_panel
from uotreg import reverse_sim as RS     # the reverse-noise data generator
import _rev_common as C           # shared data npz path/format for the baseline files

## Shared parameters
`EST_DEFAULT` / `K_DEFAULT` are the paper configs. The data knobs specific to this variant
(`delta_pre`, the graded batch strengths) live in the next cell and are echoed into every saved
npz. With the noise front-loaded, the estimator's hard times are the EARLY ones (t = 0..4), so
`h` / `budget` are the levers there.

In [ ]:
DIM        = globals().get("DIM", 10)
# ----------------------------------------------------------------------------- SMOKE
# 1 = small and fast: runs end to end on a laptop. **NOT the paper's numbers.**
# 0 = the settings used in the paper.
SMOKE      = 1
# 1 = write results/figures to `new_results/`; 0 = keep everything in memory.
# The shipped `results/` tree is never modified either way.
SAVE = globals().get("SAVE", 0)
# The estimated clouds, trajectories and metrics are a few MB per seed. The trained generators are
# (~20 MB per seed) are needed by nothing downstream, so they are skipped unless SAVE_HEAVY = 1.
SAVE_HEAVY = 0
DEVICE     = globals().get("DEVICE", "cpu")
N_PER_TIME = 200 if SMOKE else 3000        # cells per snapshot
TRAJ_T     = C.TRAJ_T                      # times 0..8 (5 snapshots in stage 1, split ~t=4)
EST_T      = C.EST_T                       # times we ESTIMATE: 1..8. t=0 is the trajectory's
#                                            starting cloud and is taken OBSERVED, so it is
#                                            never estimated (matches the bifurcation file).
STD_SPLIT  = "none" if DIM == 2 else {4: "std", 5: "std", 6: "std", 7: "std", 8: "std", "default": "std"}
RESULTS    = P.results("batcheffectreverse")                       # read
RESULTS_W  = P.results("batcheffectreverse", write=True)           # write (only when SAVE)
GENDIR     = os.path.join(RESULTS, "generators")               # read
GENDIR_W   = os.path.join(RESULTS_W, "generators")             # write
TAG        = f"reverse_d{DIM}_new" + ("_quick" if SMOKE else "")
METHODS    = ["ours: UOT maps",                  # composed UOT maps (the paper's trajectory method)
              "ours: flow OT-CFM unshared",      # flow, minibatch-OT coupling, field per interval
              "ours: flow UOT-map unshared"]     # flow trained against OUR maps (chain reused, fit once)
MK = [("adher_labeled", "adher*"), ("adher_indiv", "indiv"), ("w2_path", "w2_path"),
      ("spread_post", "spread"), ("roughness", "rough"), ("balanceA", "balA")]

EST_DEFAULT = dict(h=1.0, tau=5.0, budget=(12 if SMOKE else 100), std_mode=STD_SPLIT,
                   gen_hidden=100, gen_layers=4,
                   d_iters=(15 if SMOKE else 40), t_iters=(5 if SMOKE else 20), g_iters=(15 if SMOKE else 50),
                   n_gen=(200 if SMOKE else 3000), n_start=(40 if SMOKE else 120))
K_DEFAULT   = dict(traj_hidden=(64 if SMOKE else 256), map_layers=5,          # UOT maps
                   d_iters=(20 if SMOKE else 250), t_iters=(5 if SMOKE else 100),
                   flow_hidden=(64 if SMOKE else 128), flow_layers=4,         # flows (hidden 128)
                   flow_iters=(300 if SMOKE else 3000),                       # <- tunable
                   flow_batch=(64 if SMOKE else 256),                         # <- tunable
                   flow_lr=1e-3, flow_sigma=0.0, flow_n_per=20, flow_field="mlp", flow_seed=0,
                   tau=5.0, uot_seed=0)                                       # UOT maps reproducible
SEEDS      = list(C.SEEDS)[:1] if SMOKE else list(C.SEEDS)   # SMOKE: one seed

## The data knobs
One dict for the per-time batch strength (keys are snapshot indices, `"default"` covers the rest)
and one number for the trunk margin. Defaults live in `_rev_common.py` so the baseline files read
the same values; after editing, re-run `check_profile()` and `show_snapshots()` below.

In [ ]:
STRENGTH_SCHEDULE = {
    0: 0.5, 1: 1.5, 2: 1.5,     # first stage: the noisy one
    3: 1.0,                       # tapering into the split
    4: 0.65,                       # the split time itself (geometrically already stage 2)
    "default": 0.25,               # t >= 5: the clean second stage
}
DELTA_PRE = 0.6                    # trunk margin between the two Gaussians (bifurcation: 0.25)

DATA = dict(delta_pre=DELTA_PRE, schedule=STRENGTH_SCHEDULE)
print("batch-effect schedule (t=0..9):",
      " ".join(f"t{t}={v:.2f}" for t, v in enumerate(RS.schedule_list(STRENGTH_SCHEDULE))))
print(f"trunk margin delta_pre = {DELTA_PRE}   (bifurcation: 0.25, flat strength 0.5)")
RS.check_determinism(seed=0, schedule=STRENGTH_SCHEDULE)

In [ ]:
EST, TRAJS, FULL, ROWS = {}, {}, {}, {}   # seed -> estimate / 2-D trajs / full-d trajs / rows

_gen_path = lambda seed, t, write=False: os.path.join(GENDIR_W if write else GENDIR,
                                                     f"{TAG}_seed{seed}_G_t{t}.pth")
_musd_path = lambda seed, write=False: os.path.join(GENDIR_W if write else GENDIR,
                                                   f"{TAG}_seed{seed}_musd.npy")


def make_G(seed, n_per_time=None):
    """The data for a seed -- one place, so Section 1, the reload and the baselines all agree."""
    return RS.make_reverse_noise(seed=seed, n_per_time=(n_per_time or N_PER_TIME), dim=DIM, **DATA)


def _musd(observed, mode, dim):
    """The pooled mean/std `U.estimate` standardizes by -- needed to un-normalize a saved generator's
    draws back to the original coordinates (`std_mode` is per-time, so this is stored per time)."""
    if mode in ("std", "iso"):
        pool = np.concatenate([np.asarray(o, np.float32) for o in observed], 0)
        mu, sd = pool.mean(0), pool.std(0) + 1e-6
        if mode == "iso":
            sd = (pool.std(0).mean() + 1e-6) * np.ones(dim, np.float32)
        return np.asarray(mu, np.float32), np.asarray(sd, np.float32)
    return np.zeros(dim, np.float32), np.ones(dim, np.float32)


def estimate_seed(seed, cfg, viz=True, save=None):
    """Section-1: generate the seed's reverse-noise data, export the shared npz, estimate the series."""
    save = SAVE if save is None else save
    G = make_G(seed)
    raw = [np.asarray(G.observed[t], np.float32) for t in TRAJ_T]
    times = np.array([float(G.tlist[t]) for t in TRAJ_T], np.float32)
    X0 = raw[0][:cfg["n_start"]]
    labels0 = (np.asarray(G.labels[TRAJ_T[0]])[:cfg["n_start"]] if G.labels is not None else np.array([]))
    np.savez(C.data_path(DIM, seed, SMOKE, write=True), raw_series=np.stack(raw), times=times, X0=X0,
             labels0=labels0, dim=DIM, seed=seed, n_per=N_PER_TIME, n_start=cfg["n_start"],
             delta_pre=DATA["delta_pre"], schedule=C.schedule_to_json(DATA["schedule"]))
    t0 = time.time(); est, musd = [], []
    for t in EST_T:                       # t=0 is observed, never estimated
        mode = U.resolve_std(cfg["std_mode"], t)
        s, e = U.estimate(G.observed, G.tlist, query_time=t, dim=DIM, h=cfg["h"], tau=cfg["tau"],
                          budget=cfg["budget"], std_mode=mode,
                          gen_hidden=cfg["gen_hidden"], gen_layers=cfg["gen_layers"],
                          d_iters=cfg["d_iters"], t_iters=cfg["t_iters"], g_iters=cfg["g_iters"],
                          n_gen=cfg["n_gen"], seed=seed, device=DEVICE, return_estimator=True)
        est.append(np.asarray(s)); musd.append(np.stack(_musd(G.observed, mode, DIM)))
        if save and SAVE_HEAVY:                       # the trained generator (~2.5 MB per time)
            os.makedirs(GENDIR_W, exist_ok=True)
            e.save(_gen_path(seed, t, write=True))          # keep the trained generator -> re-draw any N later
    # the series the trajectory fitter sees: OBSERVED t=0, then the estimates for t=1..8
    s1 = dict(est_series=est, ours_series=[raw[0]] + est, raw_series=raw, est_t=list(EST_T),
              X0=X0, labels0=(labels0 if labels0.size else None))
    assert len(s1["ours_series"]) == len(TRAJ_T), "series/grid mismatch"
    EST[seed] = dict(G=G, s1=s1, cfg=cfg)
    if save:
        os.makedirs(GENDIR_W, exist_ok=True)
        np.save(_musd_path(seed, write=True), np.stack(musd))                          # (T, 2, dim)
        np.savez(os.path.join(RESULTS_W, f"{TAG}_seed{seed}_est.npz"),
                 est_series=np.stack(est), raw_series=np.stack(raw), est_t=np.array(EST_T),
                 traj_t=np.array(TRAJ_T),
                 X0=X0, labels0=labels0, times=times, musd=np.stack(musd),
                 n_gen=cfg["n_gen"], n_start=cfg["n_start"], dim=DIM, seed=seed, n_per=N_PER_TIME,
                 cfg=json.dumps({k: v for k, v in cfg.items() if k != "std_mode"}),
                 delta_pre=DATA["delta_pre"], schedule=C.schedule_to_json(DATA["schedule"]))
    print(f"[seed {seed}] estimate ({time.time()-t0:.0f}s)  per-time W2 est|raw   "
          f"(batch strength {RS.strength_at(TRAJ_T[0], DATA['schedule'])} at t={TRAJ_T[0]} -> "
          f"{RS.strength_at(TRAJ_T[-1], DATA['schedule'])} at t={TRAJ_T[-1]}):")
    tr0 = np.asarray(G.truth(TRAJ_T[0]))
    print(f"   t={TRAJ_T[0]}: {'--':>6} | {w2(raw[0], tr0):6.3f}   (observed, not estimated)")
    for i, t in enumerate(EST_T):
        tr = np.asarray(G.truth(t)); we, wr = w2(est[i], tr), w2(raw[TRAJ_T.index(t)], tr)
        print(f"   t={t}: {we:6.3f} | {wr:6.3f}" + ("  <-- est WORSE" if we > wr + 1e-6 else ""))
    if save:
        print(f"   saved {TAG}_seed{seed}_est.npz -> {os.path.relpath(RESULTS_W, P.REPO)}")
    if viz:
        show_noisy_data(G, TRAJ_T); plt.suptitle(f"seed {seed}: observed vs truth (reverse noise)", y=1.02); plt.show()
        show_estimates(G, est, EST_T, w2_fn=w2); plt.suptitle(f"seed {seed}: estimate vs truth", y=1.02); plt.show()
    return s1


def fit_seed(seed, K, viz=True, save=None):
    """Section-2: fit ours UOT maps / flow OT-CFM unshared / flow UOT-map unshared on the cached
    estimate + score + viz. The UOT map chain is fit ONCE and reused as the 'UOT-map' flow coupling."""
    save = SAVE if save is None else save
    assert seed in EST, f"run estimate_seed({seed}) first"
    G, s1 = EST[seed]["G"], EST[seed]["s1"]
    t0 = time.time()
    # `project=False` -> AMBIENT-dim trajectories. We project ourselves, so one fit yields both the
    # full-d paths (needed for the full-dimensional W2, `w2_{DIM}d`) and the 2-D ones the panels use.
    models = U.fit_trajectories(G, s1["ours_series"], s1["raw_series"], None, TRAJ_T, K,
                                methods=METHODS, device=DEVICE, return_models=True, project=False)
    full = {m: np.asarray(models[m](s1["X0"]), np.float32) for m in METHODS}   # (T, N, DIM)
    trajs = {m: G.project_traj(f) for m, f in full.items()}                    # (T, N, 2) for viz
    TRAJS[seed], FULL[seed] = trajs, full
    ROWS[seed] = TM.report(G, trajs, TRAJ_T, s1["labels0"], trajs_full=full)
    print(f"[seed {seed}] fit {len(METHODS)} methods ({time.time()-t0:.0f}s)")
    if not save:
        print(f"[seed {seed}] SAVE=0 -- trajectories kept in memory only")
        return trajs
    os.makedirs(RESULTS_W, exist_ok=True)
    np.savez(os.path.join(RESULTS_W, f"{TAG}_seed{seed}_trajs.npz"),
             **{m.replace(' ', '_').replace(':', ''): np.asarray(t) for m, t in trajs.items()},
             # the FULL-dimensional paths -- what `w2_{DIM}d` and every cross-method reader need.
             # Section 2 used to save the 2-D projection only, which made the full-d W2 unavailable.
             **{"full_" + m.replace(' ', '_').replace(':', ''): f for m, f in full.items()},
             labels0=(s1["labels0"] if s1["labels0"] is not None else np.array([])),
             traj_t=np.array(TRAJ_T), dim=DIM)
    with open(os.path.join(RESULTS_W, f"{TAG}_seed{seed}_metrics.json"), "w") as f:
        json.dump({m: {k: (None if isinstance(v, float) and np.isnan(v) else float(v))
                       for k, v in r.items()} for m, r in ROWS[seed].items()}, f, indent=2)
    if viz:
        plot_trajectory_panel(G, trajs, max_cells=60, title=f"[reverse-noise] d={DIM} seed {seed}"); plt.show()
    return ROWS[seed]


print(f"[REVERSE-NOISE d={DIM}] SMOKE={SMOKE} device={DEVICE} n_per_time={N_PER_TIME}  TAG={TAG}")
print(f"  save -> {os.path.relpath(RESULTS_W, P.REPO)} (when SAVE=1)  METHODS={METHODS}")

## Sanity check the reversed noise profile
The per-time distance between the observed snapshot and the truth should be LARGE for t <= 4 and
small after; also prints how often the branch order survives the early noise.

In [ ]:
def check_profile(seeds=range(10), n_per_time=500, show=True):
    from uotreg import simulation as _sim
    rows_rev, rows_bif, flips = [], [], 0
    for s in seeds:
        Gr = RS.make_reverse_noise(seed=s, n_per_time=n_per_time, dim=2, **DATA)
        Gb = _sim.make_bifurcation(seed=s, strength=0.5, n_per_time=n_per_time, dim=2)
        rows_rev.append([w2(np.asarray(Gr.observed[t]), np.asarray(Gr.clean[t])) for t in TRAJ_T])
        rows_bif.append([w2(np.asarray(Gb.observed[t]), np.asarray(Gb.clean[t])) for t in TRAJ_T])
        for t in TRAJ_T:                                   # branch order in the observed cloud
            o, lab = np.asarray(Gr.observed[t]), np.asarray(Gr.labels[t])
            flips += int(o[lab == 0, 1].mean() <= o[lab == 1, 1].mean())
    rev, bif = np.array(rows_rev), np.array(rows_bif)
    print(f"W2(observed, truth) per time, mean over {len(rev)} seeds:")
    print("   t        " + " ".join(f"{t:>7d}" for t in TRAJ_T))
    print("   reverse  " + " ".join(f"{v:7.2f}" for v in rev.mean(0)))
    print("   bifurc.  " + " ".join(f"{v:7.2f}" for v in bif.mean(0)))
    print("   batch strength: " + " ".join(f"t{t}={v:.2f}" for t, v in
          enumerate(RS.schedule_list(DATA["schedule"]))) + "   (bifurcation: 0.5 at every time)")
    print(f"   branch order flipped in {flips}/{len(rev)*len(TRAJ_T)} observed snapshots "
          f"(delta_pre={DATA['delta_pre']})")
    if show:
        fig, ax = plt.subplots(figsize=(5.2, 3.2), dpi=140)
        ax.plot(TRAJ_T, rev.mean(0), "o-", label="reverse noise (this file)")
        ax.plot(TRAJ_T, bif.mean(0), "s--", c="0.5", label="bifurcation (Section 6.2)")
        ax.axvline(4.5, c="0.8", lw=1)
        ax.set_xlabel("t"); ax.set_ylabel("W2(observed, truth)"); ax.legend(); fig.tight_layout(); plt.show()
    return rev, bif


check_profile()

## What the data and the manifold look like
Observed cells (orange) over the clean truth (grey) with the true branch curves dashed, one panel
per t: at t = 0..2 the cloud is visibly thrown off the truth, from t = 5 it sits on it.

In [ ]:
def show_snapshots(seed=0, n_per_time=400, dim=None, both=True):
    """The per-time strip: observed (orange) over clean truth (grey) + branch curves."""
    from uotreg import simulation as _sim
    dim = dim or DIM
    Gr = RS.make_reverse_noise(seed=seed, n_per_time=n_per_time, dim=dim, **DATA)
    show_noisy_data(Gr, TRAJ_T)
    plt.suptitle(f"reverse noise, seed {seed}: observed vs clean truth  (batch strength "
                 + ", ".join(f"t{t}:{RS.strength_at(t, DATA['schedule']):.2f}" for t in TRAJ_T)
                 + ")", y=1.03); plt.show()
    if both:
        Gb = _sim.make_bifurcation(seed=seed, strength=0.5, n_per_time=n_per_time, dim=dim)
        show_noisy_data(Gb, TRAJ_T)
        plt.suptitle(f"bifurcation (Section 6.2), seed {seed}: the original profile for contrast",
                     y=1.03); plt.show()


show_snapshots(SEEDS[0])

## Section 1: distribution estimation
Each seed's fit saves the estimated clouds + generators and exports the shared baseline data npz.

In [ ]:
for k in SEEDS:      # per-seed override: estimate_seed(k, dict(EST_DEFAULT, budget=150, h=1.5))
    estimate_seed(k, dict(EST_DEFAULT))

## Section 2: trajectory fitting
Fits on the seed's cached estimate (run its Section 1 first).

In [ ]:
for k in SEEDS:
    fit_seed(k, dict(K_DEFAULT))

## Metrics: per-seed + aggregate

In [ ]:
seeds = sorted(ROWS)
print(f"[reverse-noise d={DIM}] fitted seeds: {seeds}\n")
for s in seeds:                                  # per-seed tables
    print(f"--- seed {s} ---")
    print("   " + f"{'method':28s} " + " ".join(f"{h:>8}" for _, h in MK))
    for m in METHODS:
        print("   " + f"{m:28s} " + " ".join(
            ("     n/a" if (isinstance(ROWS[s][m][k], float) and np.isnan(ROWS[s][m][k])) else f"{ROWS[s][m][k]:8.3f}")
            for k, _ in MK))
    print()

agg = {}
print(f"[reverse-noise d={DIM}] AGGREGATE over {len(seeds)} seeds {seeds} (mean +/- std):")
print("   " + f"{'method':28s} " + " ".join(f"{h:>14}" for _, h in MK))
for m in METHODS:
    agg[m] = {}
    row = []
    for k, _ in MK:
        vals = [ROWS[s][m][k] for s in seeds
                if not (isinstance(ROWS[s][m][k], float) and np.isnan(ROWS[s][m][k]))]
        mu, sd = (float(np.mean(vals)), float(np.std(vals))) if vals else (float("nan"), 0.0)
        agg[m][k] = {"mean": mu, "std": sd, "n": len(vals)}
        row.append(f"{mu:6.3f}+-{sd:<5.3f}")
    print("   " + f"{m:28s} " + " ".join(row))

if SAVE:
    os.makedirs(RESULTS_W, exist_ok=True)
    with open(os.path.join(RESULTS_W, f"{TAG}_agg_metrics.json"), "w") as f:
        json.dump({"seeds": seeds, "data": DATA, "metrics": agg}, f, indent=2)
    print(f"\nsaved {TAG}_agg_metrics.json -> {os.path.relpath(RESULTS_W, P.REPO)}")